In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor
sys.path.append('../')
import importlib
import yaml
import torch
import os 
from pathlib import Path
import whisper
import lightning_scripts.whisper_transfer_module as whisper_module

In [ ]:
torch.cuda.is_available()

In [ ]:
## init config. Will be yaml eventually, but start as dict 
config = {}
# Overwrite config for linear classifier 
config['num_workers'] = 4
config['hparas'] = {}
config['data'] = {}
config['hparas']['batch_size'] = 192
config['hparas']['optimizer'] = "AdamW"
config['hparas']['lr'] = 0.0005
config['data']['eval_max'] = 1
config['model'] = {}
config['model']['arch_kwargs'] = {} 
config['model']['arch_kwargs']['supervised'] = False
config['model']['arch_kwargs']['num_classes'] = {"signal/word_int": 794,    
                            "signal/speaker_int": 433} 

# add task loss params to hparas 
config['hparas']['task_loss_params'] = {
    "signal/word_int":
        {"loss_type": 'crossentropyloss',
        "weight": 1.0},                                       # init loss is ~200 
    "signal/speaker_int":
        {"loss_type": 'crossentropyloss',
        "weight": 1.0}
            }
config['data']['target_keys'] = list(config['model']['arch_kwargs']['num_classes'].keys())

config['model']['whisper_model'] = 'large-v3-turbo'


In [ ]:
importlib.reload(whisper_module)

module = whisper_module.WhisperTransferModule(config=config)


In [ ]:
trainer = L.Trainer(devices=1)
trainer.fit(module)

## Dev Eval Collate Function

In [ ]:
from lightning_scripts.jsinV3DataLoader_precombined_batched import CleanSpeechInNoiseValDatasetBatched

eval_speech_h5_path = '/mnt/home/jfeather/ceph/data/training_datasets_audio/jsinV3BalancedProcessed/sr_20000/splits/train_stackedDataframeHDF_n150_VJRUH4IEPDGPNH2JZMULSQKOWYNQ6KMM.pdh5'

test_dataset = CleanSpeechInNoiseValDatasetBatched(speech_h5_path=eval_speech_h5_path,
                                        target_keys=config['data']['target_keys'],
                                        batch_size=4,
                                        )

In [ ]:
batch = test_dataset[0]

In [ ]:
audio, labels = batch
audio = module.resample_audio(audio)
audio = whisper.pad_or_trim(audio)
mel = whisper.log_mel_spectrogram(audio, n_mels=module.n_mels)



In [ ]:
mel.shape

In [ ]:
outputs = trainer.predict(module, module.val_dataloader(), return_predictions=True)

In [ ]:
top1_word = []
top1_speaker = []
top5_word = []
top5_speaker = []

for record in outputs:
    top1_word.append(record['top1']['signal/word_int'])
    top1_speaker.append(record['top1']['signal/speaker_int'])
    top5_word.append(record['top5']['signal/word_int'])
    top5_speaker.append(record['top5']['signal/speaker_int'])

In [ ]:
n_examples = len(outputs)
output_dict = {
    "word_top1_mean": torch.stack(top1_word).mean(),
    "word_top1_sem": torch.stack(top1_word).std() / np.sqrt(n_examples),
    "speaker_top1_mean": torch.stack(top1_speaker).mean(),
    "speaker_top1_sem": torch.stack(top1_speaker).std() / np.sqrt(n_examples),

    "word_top5_mean": torch.stack(top5_word).mean(),
    "word_top5_sem": torch.stack(top5_word).std() / np.sqrt(n_examples),
    "speaker_top5_mean": torch.stack(top5_speaker).mean(),
    "speaker_top5_sem": torch.stack(top5_speaker).std() / np.sqrt(n_examples),
}
output_dict = {key:val.item() for key,val in output_dict.items()}

In [ ]:
output_dict

In [ ]:
output_dict = {}
for key, task_dict in outputs.items():
    for task, scores in task_dict.items():
        n_examples = len(scores)
        task_str = task.split('/')[-1]
        output_dict[f"{task_str}_{key}_mean"] = torch.cat(scores).mean()
        output_dict[f"{task_str}_{key}_sem"] = torch.cat(scores).std() / np.sqrt(n_examples)

In [ ]:
output_vals = torch.cat([output['accuracy'] for output in outputs])
len(output_vals)

In [ ]:
output_vals.mean()

In [ ]:

output_vals.std(unbiased=True) / (output_vals.size(0) ** 0.5)

In [ ]:
import pickle
with open('eval_jsin_results/ssl_barlow_word_resnet50_hparam_set_0_linear_eval_jsin.pkl', 'rb') as handle:

    results = pickle.load(handle)

In [ ]:
results